# 12 — Capstone: Efficient LM

Train a small Mamba + Transformer + MoE and benchmark quality, speed, and memory.

In [ ]:
# Shared setup
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import matplotlib.pyplot as plt

torch.manual_seed(42)

VOCAB = 64
D = 32
SEQ = 32
N_LAYERS = 2

# Dataset: simple patterns (next-token prediction)
def make_dataset(n=2000):
    X = torch.zeros(n, SEQ, dtype=torch.long)
    for i in range(n):
        start = torch.randint(0, VOCAB//2, (1,)).item()
        step = torch.randint(1, 4, (1,)).item()
        for t in range(SEQ):
            X[i, t] = (start + t * step) % VOCAB
    return X

X_all = make_dataset()
X_train, X_val = X_all[:1600], X_all[1600:]

def train_model(model, n_steps=1000, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for step in range(n_steps):
        idx = torch.randint(0, len(X_train), (32,))
        x = X_train[idx, :-1]
        y = X_train[idx, 1:]
        logits = model(x)
        if isinstance(logits, tuple): logits = logits[0]
        loss = nn.CrossEntropyLoss()(logits.reshape(-1, VOCAB), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    return model

def eval_ppl(model):
    model.eval()
    with torch.no_grad():
        x = X_val[:100, :-1]
        y = X_val[:100, 1:]
        logits = model(x)
        if isinstance(logits, tuple): logits = logits[0]
        loss = nn.CrossEntropyLoss()(logits.reshape(-1, VOCAB), y.reshape(-1))
    return torch.exp(loss).item()

def bench_speed(model, n=100):
    x = torch.randint(0, VOCAB, (4, SEQ-1))
    model.eval()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(n): model(x)
    return (time.perf_counter() - t0) / n * 1000

def count_params(model):
    return sum(p.numel() for p in model.parameters())

print('Dataset and benchmark utilities ready')

In [ ]:
# Model 1: Transformer
class TransformerLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, D)
        layer = nn.TransformerEncoderLayer(D, nhead=4, dim_feedforward=D*4,
                                            batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=N_LAYERS)
        self.head = nn.Linear(D, VOCAB)

    def forward(self, x):
        h = self.embed(x)
        mask = nn.Transformer.generate_square_subsequent_mask(x.shape[1])
        h = self.encoder(h, mask=mask, is_causal=True)
        return self.head(h)

# Model 2: Simple SSM (S4-style)
class SSMLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, D)
        self.layers = nn.ModuleList([
            nn.GRU(D, D, batch_first=True)  # GRU as SSM proxy
            for _ in range(N_LAYERS)
        ])
        self.head = nn.Linear(D, VOCAB)

    def forward(self, x):
        h = self.embed(x)
        for layer in self.layers:
            h, _ = layer(h)
        return self.head(h)

# Model 3: MoE Transformer
class MoETransformerLM(nn.Module):
    def __init__(self, n_experts=4):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, D)
        self.attn = nn.MultiheadAttention(D, num_heads=4, batch_first=True)
        self.norm1 = nn.LayerNorm(D)
        self.norm2 = nn.LayerNorm(D)
        self.experts = nn.ModuleList([nn.Sequential(nn.Linear(D, D*2), nn.GELU(), nn.Linear(D*2, D))
                                       for _ in range(n_experts)])
        self.router = nn.Linear(D, n_experts)
        self.head = nn.Linear(D, VOCAB)
        self.n_experts = n_experts

    def moe(self, x):
        B, T, Dv = x.shape
        x_flat = x.reshape(-1, Dv)
        probs = self.router(x_flat).softmax(-1)  # (N, E)
        top2_probs, top2_idx = probs.topk(2, dim=-1)
        top2_probs = top2_probs / top2_probs.sum(-1, keepdim=True)
        out = torch.zeros_like(x_flat)
        for k in range(2):
            for e in range(self.n_experts):
                mask = (top2_idx[:, k] == e)
                if mask.sum() > 0:
                    out[mask] += top2_probs[mask, k:k+1] * self.experts[e](x_flat[mask])
        return out.reshape(B, T, Dv)

    def forward(self, x):
        h = self.embed(x)
        B, T, _ = h.shape
        mask = nn.Transformer.generate_square_subsequent_mask(T)
        h2, _ = self.attn(h, h, h, attn_mask=mask, is_causal=True)
        h = self.norm1(h + h2)
        h = self.norm2(h + self.moe(h))
        return self.head(h)

# Train all models
models = []
for ModelClass, name in [(TransformerLM, 'Transformer'), (SSMLM, 'SSM/GRU'), (MoETransformerLM, 'MoE-Trans')]:
    m = ModelClass()
    print(f'Training {name}...')
    train_model(m, n_steps=500)
    ppl = eval_ppl(m)
    spd = bench_speed(m)
    params = count_params(m)
    models.append((name, m, ppl, spd, params))
    print(f'  {name}: ppl={ppl:.1f}, speed={spd:.1f}ms, params={params}')

In [ ]:
# Final dashboard
names = [r[0] for r in models]
ppls = [r[2] for r in models]
speeds = [r[3] for r in models]
params = [r[4] for r in models]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].bar(names, ppls, color=['steelblue', 'green', 'purple'], alpha=0.8)
axes[0].set_title('Perplexity (lower = better)')
axes[0].set_ylabel('Perplexity')

axes[1].bar(names, speeds, color=['steelblue', 'green', 'purple'], alpha=0.8)
axes[1].set_title('Inference Time (ms, lower = faster)')
axes[1].set_ylabel('ms per call')

axes[2].bar(names, params, color=['steelblue', 'green', 'purple'], alpha=0.8)
axes[2].set_title('Parameter Count')
axes[2].set_ylabel('Parameters')

plt.suptitle('Efficient Architecture Capstone: Quality vs Speed vs Size')
plt.tight_layout()
plt.savefig('/tmp/capstone_efficient.png', dpi=100, bbox_inches='tight')
plt.show()
print('\nSeries 24 (Efficient Architectures & Alternative Transformers) complete.')